# 유튜브 자막 추출

자막 추출용 영상 리스트

영상
-----------------------

https://www.youtube.com/watch?v=qRwOTEc3PuU

https://www.youtube.com/watch?v=wHAzS0Gcn1o

-----------------------

쇼츠
-----------------------

https://www.youtube.com/shorts/KpunAbRUxP8

https://www.youtube.com/shorts/nklrjJRfH-Q

https://www.youtube.com/shorts/4nwv-DXRxkU

https://www.youtube.com/shorts/R9ZzxjC3YCA

https://www.youtube.com/shorts/ROFjvCezod4

https://www.youtube.com/shorts/7ZuXQ8qeEo0

https://www.youtube.com/shorts/vq6_47FLOeI

-----------------------

In [67]:
import yt_dlp
import json
from urllib.parse import urlparse, parse_qs
import os
from youtube_transcript_api.formatters import SRTFormatter, TextFormatter

In [36]:
# 불러오는 코드 부분
def get_youtube_video_info(video_url):
    ydl_opts = {
        'noplaylist': True,
        'quiet': True,
        'no_warnings': True,
    }
    
    # yt_dlp.YoutubeDL 클래스를 사용하여 객체 생성
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        video_info = ydl.extract_info(video_url, download=False)
        
        # 쇼츠 URL 처리
        if 'shorts' in video_url:
            # 쇼츠는 일반 영상과 달리 ID만 필요
            video_id = video_url.split('/shorts/')[1]
            title = video_info.get('title', 'No Title')
            upload_date = video_info.get('upload_date', 'Unknown')
            channel = video_info.get('channel', 'Unknown')
            duration = 'Unknown'
        else:
            # 일반 영상 처리
            video_id = video_info['id']
            title = video_info['title']
            upload_date = video_info['upload_date']
            channel = video_info['channel']
            duration = video_info.get('duration_string', 'Unknown')

        return {
            'video_id': video_id,
            'title': title,
            'upload_date': upload_date,
            'channel': channel,
            'duration': duration
        }

In [37]:
#url 붙여넣는 부분
video_url = 'https://www.youtube.com/watch?v=qRwOTEc3PuU&t=402s'
get_youtube_video_info(video_url)

{'video_id': 'qRwOTEc3PuU',
 'title': '역대급 설 도시락, 저속노화 식단, 빠스맛 오사쯔~ 1월 셋째 주 편의점 신제품 리뷰',
 'upload_date': '20250119',
 'channel': '맛상무',
 'duration': '9:41'}

In [4]:
#자막 가져오는 코드
def get_video_id(video_url):
    parsed_url = urlparse(video_url)
    
    # 쇼츠 형식 처리
    if parsed_url.netloc == 'www.youtube.com' and parsed_url.path.startswith('/shorts/'):
        return parsed_url.path.split('/shorts/')[1]
    
    # `youtu.be` 형식 처리
    if parsed_url.netloc == 'youtu.be':
        return parsed_url.path.lstrip('/')
    
    # 일반 유튜브 URL에서 `v` 파라미터 추출
    query_params = parse_qs(parsed_url.query)
    if 'v' in query_params:
        return query_params['v'][0][:11]
    
    raise ValueError("유효한 유튜브 URL이 아닙니다.")

## 일반 영상 자막 추출

In [5]:
#TODO 아래의 링크만 바꿔도 된다.
#자막 따기를 위한 id 추출
video_url = 'https://www.youtube.com/watch?v=qRwOTEc3PuU&t=402s'
video_id = get_video_id(video_url)

In [6]:
metadata = get_youtube_video_info(video_url)
title = metadata['title']
upload_date = metadata['upload_date']  # YYYYMMDD 형식
formatted_date = f"{upload_date[:4]}-{upload_date[4:6]}-{upload_date[6:]}"  # YYYY-MM-DD 형식
print(f"- 제목: {title}")
print(f"- 등록일: {formatted_date}")

- 제목: 역대급 설 도시락, 저속노화 식단, 빠스맛 오사쯔~ 1월 셋째 주 편의점 신제품 리뷰
- 등록일: 2025-01-19


In [7]:
#언어 인식

from youtube_transcript_api import YouTubeTranscriptApi
transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)

for transcript in transcript_list:
    print(f"- [자막언어] {transcript.language}, [자막 언어 코드] {transcript.language_code}")

- [자막언어] Korean (auto-generated), [자막 언어 코드] ko


In [8]:
# 자막 추출 및 저장
transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['ko', 'en'])
transcript_text_only = [item['text'] for item in transcript]  # 텍스트만 추출

download_folder = "./make_service_for_me"
os.makedirs(download_folder, exist_ok=True)  # 디렉터리가 없으면 생성

In [9]:
# JSON 저장
json_file = f"{download_folder}/{video_id}.json"
metadata_with_transcript = {
    'title': title,
    'upload_date': formatted_date,
    'transcript': transcript_text_only,  # 텍스트만 저장
}
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(metadata_with_transcript, f, ensure_ascii=False, indent=4)
print("- JSON 파일 경로:", json_file)

- JSON 파일 경로: ./make_service_for_me/qRwOTEc3PuU.json


--------------------

## 쇼츠자막 추출

In [73]:
#TODO 쇼츠용 id 추출 부분
# # 쇼츠용 id 추출

shorts_url = 'https://www.youtube.com/shorts/vq6_47FLOeI'
shorts_metadata = get_youtube_video_info(shorts_url)
shorts_id = shorts_metadata['video_id']
shorts_title = shorts_metadata['title']
shorts_upload_date = shorts_metadata['upload_date']
shorts_formatted_date = f"{shorts_upload_date[:4]}-{shorts_upload_date[4:6]}-{shorts_upload_date[6:]}"

In [74]:
#언어 인식

from youtube_transcript_api import YouTubeTranscriptApi
transcript_list = YouTubeTranscriptApi.list_transcripts(shorts_id)

for transcript in transcript_list:
    print(f"- [자막언어] {transcript.language}, [자막 언어 코드] {transcript.language_code}")

- [자막언어] Korean (auto-generated), [자막 언어 코드] ko


In [75]:
# 쇼츠 자막 추출
shorts_transcript = YouTubeTranscriptApi.get_transcript(shorts_id, languages=['ko', 'en'])
shorts_transcript_text_only = [item['text'] for item in shorts_transcript]

download_folder = "./make_service_for_me"
os.makedirs(download_folder, exist_ok=True)  # 디렉터리가 없으면 생성

In [76]:

# JSON 저장
shorts_json_file = f"{download_folder}/{shorts_id}.json"
shorts_metadata_with_transcript = {
    'title': shorts_title,
    'upload_date': shorts_formatted_date,
    'transcript': shorts_transcript_text_only,
}
with open(shorts_json_file, 'w', encoding='utf-8') as f:
    json.dump(shorts_metadata_with_transcript, f, ensure_ascii=False, indent=4)
print("- JSON 파일 경로:", json_file)

- JSON 파일 경로: ./make_service_for_me/4nwv-DXRxkU.json


# text 저장 - 파일 확인해보니 글자가 다 깨짐
transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['en', 'ko'])
text_formatter = TextFormatter()
text_formatted = text_formatter.format_transcript(transcript)
print(text_formatted[:100])
text_file = f"{download_folder}/{video_id}.txt"
with open(text_file, 'w') as f:
    f.write(text_formatted)
print("- TXT 파일경로:", text_file)